In [35]:
from pydantic import BaseModel
from typing import Literal
from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

In [7]:
load_dotenv(override=True)

True

In [9]:
class ConfigureModel:

    def __init__(self, model_name:str="openai/gpt-oss-20b", groq_api_key:str=os.getenv('GROQ_API_KEY')):
        self.model_name = model_name
        self.groq_api_key = groq_api_key
        self.model = None
        self._initiate_model()
    
    def _initiate_model(self):
        print(f"Trying to connect with model {self.model_name}...")
        try:
            self.model = ChatGroq(
                model = self.model_name,
                temperature = 0.7,
                groq_api_key = self.groq_api_key
            )
            print("Model connection successfull")
        
        except Exception as e:
            print(f"Error: {e}")

    def get_response(self, prompt):
        print(f"prompt sent to model, waiting for response")
        try:
            response = self.model.invoke(prompt)
            return response.content
        
        except Exception as e:
            print(f"Error: {e}")        

In [10]:
model_configurer = ConfigureModel()

Trying to connect with model openai/gpt-oss-20b...
Model connection successfull


## PydanticOutputParser

In [15]:
class Feedback(BaseModel):
    sentiment:Literal['positive', 'negative']

In [36]:
pydantic_parser = PydanticOutputParser(pydantic_object=Feedback)
str_parser = StrOutputParser()

## Prompts

In [ ]:
prompt1 = PromptTemplate(
    template = "Give sentiment of this feedback {feedback}\n {format_instruction}",
    input_variables = ['feedback'],
    partial_variables = {'format_instruction' : pydantic_parser.get_format_instructions()}
)

prompt2 = PromptTemplate(
    template = "Give a usual response sentence of this feedback: {feedback}",
    input_variables = ['feedback']
)

## Creating Chains

In [23]:
chain1 = prompt1 | model_configurer.model | pydantic_parser

In [40]:
# lambda functions ensures that x['sentiment'] is checked once chain1 has executed
conditional_chain = RunnableBranch(
    (lambda x:x.sentiment =='positive', prompt2 | model_configurer.model | str_parser),
    (lambda x:x.sentiment=='negative', prompt2 | model_configurer.model | str_parser),
    RunnableLambda(lambda x : "could not find x")
)

In [41]:
final_chain = chain1 | conditional_chain

In [42]:
result = final_chain.invoke({'feedback':'The book is amazing'})

In [43]:
print(result)

Thank you for your positive feedback—we’re thrilled to hear you enjoyed your experience!
